# key 引数を使い複雑な基準でソートする

組み込み型 list には、さまざまな基準に基づいて list の要素を順序に従って整列する sort メソッドがあります。

In [1]:
numbers = [93, 86, 11, 68, 70]
numbers.sort()
print(numbers)  # [11, 68, 70, 86, 93

[11, 68, 70, 86, 93]


sort メソッドは、自然な順序のあるほとんどの組み込み型（文字列、浮動小数点など）で動作します。

オブジェクトではどうでしょうか。例えば、建設現場で使う様々な工具を表すクラスを、インスタンスを出力できるよう \_\_repr\_\_ メソッドを定義しました。

In [2]:
class Tool:
  def __init__(self, name, weight):
    self.name = name
    self.weight = weight

  def __repr__(self):
    return f'Tool({self.name!r}, {self.weight})'

tools = [
  Tool('level', 3.5),
  Tool('hammer', 1.25),
  Tool('wrench', 0.5),
  Tool('screwdriver', 0.25),
]

この型のオブジェクトのソートは、クラスで定義されていない比較のための特殊メソッドを sort メソッドが呼び出そうとするので失敗します。

In [3]:
tools.sort()

TypeError: '<' not supported between instances of 'Tool' and 'Tool'

クラスに整数のような自然がない場合には、必要な特殊メソッドを定義して、追加のパラメータがなくなっても sort が動作する必要があり、自然な順序の定義だけでは意味がありません。

しばしば、オブジェクトの属性値によってソートする場合があります。このユースケースをサポートするために、sort メソッドには関数を引数として期待する key パラメータが使えます。key 関数には単一引数としてソートされるリストの要素が渡されます。戻り値は、ソートのために使用する比較可能な（自然な順序を持つ）値でなければなりません。

次のコードでは、Tool オブジェクトのリストを名前の英字順にソートできる key パラメータのためのお関数を lambda キーワードを使って定義します。

In [6]:
print('Unsorted:', repr(tools))
# lambda 引数: 戻り値
# def name_key(x):
#     return x.name
# と同じ意味
tools.sort(key=lambda x: x.name)
print('\nSorted:', tools)

Unsorted: [Tool('hammer', 1.25), Tool('level', 3.5), Tool('screwdriver', 0.25), Tool('wrench', 0.5)]

Sorted: [Tool('hammer', 1.25), Tool('level', 3.5), Tool('screwdriver', 0.25), Tool('wrench', 0.5)]


key パラメータとして渡されるラムダ関数の中で要素の属性に、ここで示したように（シーケンス、タプル、辞書では）要素のインデックス、あるいは、他の式を用いてアクセスできます。

文字列のような基本型では、key 関数を使ってソートの前に値の変換ができます。例えば、次のコードでは、（自然な英字順では、大文字が小文字の前に来るので）大文字小文字を無視した英字順になるとうに list の中の要素の名前に lower メソッドを適用しています。

In [7]:
places = ['home', 'work', 'New York', 'Paris', 'London']
places.sort()
print('Case sensitive:  ', places)
places.sort(key=lambda s: s.lower())
print('Case insensitive:', places)

Case sensitive:   ['London', 'New York', 'Paris', 'home', 'work']
Case insensitive: ['home', 'London', 'New York', 'Paris', 'work']


ソートに複数の基準を使う必要が生じることがあります。例えば、工具のリストがあり、まず重さ、次に名前でソートしたいとします。どのようにすればよいでしょうか。

In [8]:
power_tools = [
  Tool('drill', 4),
  Tool('circular saw', 5),
  Tool('jigsaw', 40),
  Tool('sander', 4),
]

Python で最も単純な解放は、tuple 型を用いることです。tuple は任意の Python 値の変更不能なシーケンスです。tuple はデフォルトで比較可能で、自然な順序があり、sort メソッドが必要とする \_\_it\_\_ のような特殊メソッドを備えています。tuple 同士では、tuple の各位置の要素について実装されている特殊メソッドで順に比較していきます。

ある工具が別の道具よりも重い場合にこれがどうなるかを次に示します。

In [10]:
saw = (5, 'circular saw')
jigsaw = (40, 'jigsaw')
assert not (saw > jigsaw) # 期待通り

タプルの最初の位置の要素（この場合は重さ）が等しいと、タプルの比較は次の位置の要素の比較に移ります。

In [11]:
drill = (4, 'drill')
sander = (4, 'sander')
assert drill[0] == sander[0]  # 重さは同じ
assert drill[1] < sander[1]   # 名前で比較される
assert drill < sander         # よって drill が前に来る

この tuple の比較方式を使って、工具をまずその重さで次にその名前でソートできます。次のコードでは、優先順位順にソートする 2つの属性の特性を返す key 関数を定義しています。

In [12]:
power_tools.sort(key=lambda tool: (tool.weight, tool.name))
print('\nSorted by weight, then name:', power_tools)


Sorted by weight, then name: [Tool('drill', 4), Tool('sander', 4), Tool('circular saw', 5), Tool('jigsaw', 40)]


list 型の sort メソッドは、key 関数が互いが等しいという値を返したときには、入力リストでの順番をほじします。
これは、同じリストに sort を複数回呼び出して、異なる基準を組み合わせられるということを意味します。次のコードでは、上で行ったように weight が降順、name が昇順になるソートを sort を別々に 2回呼び出して行いました。

In [14]:
power_tools.sort(key=lambda tool: tool.name) # 名前で昇順ソート
power_tools.sort(key=lambda tool: tool.weight, reverse=True) # 重さで降順

print(power_tools)

[Tool('jigsaw', 40), Tool('circular saw', 5), Tool('drill', 4), Tool('sander', 4)]


なぜこれがうまくいくかを理解するには、sort の最初の呼び出しが名前を英字順にすることに注意します。

In [15]:
power_tools.sort(key=lambda tool: tool.name)
print(power_tools)

[Tool('circular saw', 5), Tool('drill', 4), Tool('jigsaw', 40), Tool('sander', 4)]


2番目の sort が weight の降順で呼ばれると、'sander' と 'drill が同じ重さの 4 であることに気づきます。
これにより、sort メソッドは、この 2要素を元の list に現れていたのと同じ順序になるように置くので、name の相対順序は昇順に保たれます。

In [16]:
power_tools.sort(key=lambda tool: tool.weight, reverse=True)
print(power_tools)

[Tool('jigsaw', 40), Tool('circular saw', 5), Tool('drill', 4), Tool('sander', 4)]


これと同じ方式が、さまざまな種類のソート基準を好きな方向に組み合わせるのに使えます。最終の list で保持するものを逆のシーケンスでソートするように確認すればよいだけです。この例では、weight を降順に、name を照準にしたいので、name をまずソートしてから、次に weight をソートする必要がありました。

つまり、key 関数で tuple を返し、マイナス単項演算子を使って異なるソート順を混在させる手法の方が読みやすく、必要なコード量も少なく済みます。
複数回ソートを呼び出すのは絶対に櫃お湯な場合のみにすることをお勧めします。

## 覚えておくこと

- list 型の sort メソッドは、リストの内容を文字列、整数、タプルなどの自然な順序で並べ替えるのに使う。
- sort メソッドは、特殊メソッドを使って順序付けするメソッドを定義しないとオブジェクトで動作しないが、それは一般的にはあまりない
- sort メソッドの key パラメータを使い、list の各要素をソートする値を返すヘルパー関数を与えることができる
- key 関数で tuple を返すことにより、複数のソート基準を組み合わせることができる。単項マイナス演算子が、その型で許されるソート順を逆転するのに使える
- マイナス演算子を使えない型では、sort メソッドをさまざまな key 関数と reverse 値に対して、最も低いランクの sort 呼び出しから最も高いランクの sort 呼び出しまで順に複数回呼び出すことで組み合わせたソートができる